# Problem 1: RNN, LSTM, and GRU on Sequences
**Student Name:** Gonuguntla Anand  
**Student ID:** 2026ajo5503 (Ablation Variant: 3 - Odd: LSTM Forget-Gate Bias Init 0 vs 1)  
**Course:** Deep Neural Networks (M.Tech, BITS Pilani WILP)  

This notebook implements vanilla RNN, LSTM, and GRU cells from scratch using fundamental PyTorch tensor operations, rigorously validates them against library cells, derives parameter counts analytically, and benchmarks performance across real-world sensor data (UCI HAR) and the synthetic Adding Problem.


In [ ]:
import os
import math
import time
import random
import urllib.request
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Active Device: {device}")

SEEDS = [42, 123, 999]
HIDDEN_DIM = 64
FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

RESULTS_CSV = "../results.csv"


## Task 1: Data Pipelines
We implement:
1. **UCI HAR Data Pipeline:** 128 steps × 9 channels across 6 activity classes. Validation holds out 4 of the 21 training subjects using `np.random.default_rng(seed).choice`. Per-channel mean and std are calculated **strictly from the training subjects**.
2. **Adding Problem Generator:** Two marked steps with uniform noise input and exact sum target.


In [ ]:
# Download & Extract UCI HAR Dataset if not present
har_url = 'https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip'
if not os.path.exists('UCI HAR Dataset'):
    if not os.path.exists('har.zip'):
        print("Downloading UCI HAR dataset...")
        urllib.request.urlretrieve(har_url, 'har.zip')
    with zipfile.ZipFile('har.zip') as z:
        z.extractall('.')
    if os.path.exists('UCI HAR Dataset.zip'):
        with zipfile.ZipFile('UCI HAR Dataset.zip') as z:
            z.extractall('.')

CHANNELS = [
    'body_acc_x', 'body_acc_y', 'body_acc_z',
    'body_gyro_x', 'body_gyro_y', 'body_gyro_z',
    'total_acc_x', 'total_acc_y', 'total_acc_z'
]
CLASS_NAMES = ['Walking', 'Walking Upstairs', 'Walking Downstairs', 'Sitting', 'Standing', 'Laying']

def load_har_data(base_path='UCI HAR Dataset'):
    def load_split(split):
        data = []
        for ch in CHANNELS:
            arr = np.loadtxt(os.path.join(base_path, split, 'Inertial Signals', f'{ch}_{split}.txt'))
            data.append(arr)
        X = np.stack(data, axis=-1)  # (N, 128, 9)
        y = np.loadtxt(os.path.join(base_path, split, f'y_{split}.txt')).astype(int) - 1  # 0-indexed
        subj = np.loadtxt(os.path.join(base_path, split, f'subject_{split}.txt')).astype(int)
        return X, y, subj
    X_train_full, y_train_full, subj_train_full = load_split('train')
    X_test, y_test, subj_test = load_split('test')
    return X_train_full, y_train_full, subj_train_full, X_test, y_test, subj_test

X_train_full, y_train_full, subj_train_full, X_test, y_test, subj_test = load_har_data()
print(f"Loaded Train: {X_train_full.shape}, Test: {X_test.shape}")
assert X_train_full.shape == (7352, 128, 9), "Train shape mismatch"
assert X_test.shape == (2947, 128, 9), "Test shape mismatch"


In [ ]:
def get_har_split(X_train_full, y_train_full, subj_train_full, X_test, y_test, seed):
    train_subjects = np.unique(subj_train_full)
    rng = np.random.default_rng(seed)
    val_subjs = rng.choice(train_subjects, size=4, replace=False)
    val_mask = np.isin(subj_train_full, val_subjs)
    train_mask = ~val_mask

    X_train = X_train_full[train_mask]
    y_train = y_train_full[train_mask]
    X_val = X_train_full[val_mask]
    y_val = y_train_full[val_mask]

    # Per-channel mean & std strictly from training subjects
    mean = np.mean(X_train, axis=(0, 1), keepdims=True)
    std = np.std(X_train, axis=(0, 1), keepdims=True) + 1e-8

    X_train_norm = (X_train - mean) / std
    X_val_norm = (X_val - mean) / std
    X_test_norm = (X_test - mean) / std

    return (X_train_norm, y_train), (X_val_norm, y_val), (X_test_norm, y_test)


In [ ]:
# Plot and save Class Balance
fig, ax = plt.subplots(figsize=(8, 4.5))
train_counts = np.bincount(y_train_full, minlength=6)
test_counts = np.bincount(y_test, minlength=6)
x = np.arange(6)
width = 0.35
ax.bar(x - width/2, train_counts, width, label='Train Set (7,352)', color='#1f77b4', edgecolor='black')
ax.bar(x + width/2, test_counts, width, label='Test Set (2,947)', color='#ff7f0e', edgecolor='black')
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, rotation=25, ha='right')
ax.set_ylabel('Number of Windows')
ax.set_title('UCI HAR Dataset Class Distribution Across Splits')
ax.legend(frameon=True)
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'p1_class_balance.png'))
plt.show()


In [ ]:
# Exact generator function from assignment
def make_adding(n, T, seed):
    rng = np.random.default_rng(seed)
    x = np.zeros((n, T, 2), dtype=np.float32)
    x[:, :, 0] = rng.random((n, T))
    i1 = rng.integers(0, T // 2, n)
    i2 = rng.integers(T // 2, T, n)
    x[np.arange(n), i1, 1] = 1
    x[np.arange(n), i2, 1] = 1
    y = x[np.arange(n), i1, 0] + x[np.arange(n), i2, 0]
    return x, y.astype(np.float32)


In [ ]:
# Unit test for make_adding generator
x_t, y_t = make_adding(500, 100, seed=42)
assert x_t.shape == (500, 100, 2), "Shape mismatch"
assert y_t.shape == (500,), "Target shape mismatch"

for i in range(500):
    markers = np.where(x_t[i, :, 1] == 1)[0]
    assert len(markers) == 2, f"Expected 2 markers, found {len(markers)}"
    assert markers[0] < 50, f"Marker 1 ({markers[0]}) not in first half"
    assert markers[1] >= 50, f"Marker 2 ({markers[1]}) not in second half"
    calc_sum = x_t[i, markers[0], 0] + x_t[i, markers[1], 0]
    assert np.isclose(y_t[i], calc_sum, atol=1e-5), "Sum mismatch"

print("✓ make_adding unit tests passed!")


## Task 2: RNN, LSTM, and GRU Cells from Scratch
We implement recurrent cells using only low-level linear tensor operations and activations, without importing `nn.RNN`, `nn.LSTM`, or `nn.GRU`.

### Mathematical Formulations:
- **Vanilla RNN:** $h_t = \tanh(W_{ih} x_t + b_{ih} + W_{hh} h_{t-1} + b_{hh})$
- **LSTM:**
  $$\begin{aligned}
  i_t &= \sigma(W_{ii} x_t + b_{ii} + W_{hi} h_{t-1} + b_{hi}) \\
  f_t &= \sigma(W_{if} x_t + b_{if} + W_{hf} h_{t-1} + b_{hf}) \\
  g_t &= \tanh(W_{ig} x_t + b_{ig} + W_{hg} h_{t-1} + b_{hg}) \\
  o_t &= \sigma(W_{io} x_t + b_{io} + W_{ho} h_{t-1} + b_{ho}) \\
  c_t &= f_t \odot c_{t-1} + i_t \odot g_t \\
  h_t &= o_t \odot \tanh(c_t)
  \end{aligned}$$
- **GRU:**
  $$\begin{aligned}
  r_t &= \sigma(W_{ir} x_t + b_{ir} + W_{hr} h_{t-1} + b_{hr}) \\
  z_t &= \sigma(W_{iz} x_t + b_{iz} + W_{hz} h_{t-1} + b_{hz}) \\
  n_t &= \tanh(W_{in} x_t + b_{in} + r_t \odot (W_{hn} h_{t-1} + b_{hn})) \\
  h_t &= (1 - z_t) \odot n_t + z_t \odot h_{t-1}
  \end{aligned}$$


In [ ]:
class MyRNNCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.weight_ih = nn.Parameter(torch.empty(hidden_size, input_size))
        self.weight_hh = nn.Parameter(torch.empty(hidden_size, hidden_size))
        self.bias_ih = nn.Parameter(torch.empty(hidden_size))
        self.bias_hh = nn.Parameter(torch.empty(hidden_size))
        self.reset_parameters()

    def reset_parameters(self):
        stdv = 1.0 / math.sqrt(self.hidden_size)
        for p in self.parameters():
            nn.init.uniform_(p, -stdv, stdv)

    def forward(self, x, h):
        # x: (B, input_size), h: (B, hidden_size) -> returns (B, hidden_size)
        return torch.tanh(F.linear(x, self.weight_ih, self.bias_ih) + 
                          F.linear(h, self.weight_hh, self.bias_hh))


class MyLSTMCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.weight_ih = nn.Parameter(torch.empty(4 * hidden_size, input_size))
        self.weight_hh = nn.Parameter(torch.empty(4 * hidden_size, hidden_size))
        self.bias_ih = nn.Parameter(torch.empty(4 * hidden_size))
        self.bias_hh = nn.Parameter(torch.empty(4 * hidden_size))
        self.reset_parameters()

    def reset_parameters(self):
        stdv = 1.0 / math.sqrt(self.hidden_size)
        for p in self.parameters():
            nn.init.uniform_(p, -stdv, stdv)

    def forward(self, x, state):
        # x: (B, input_size), state: (h, c) each (B, hidden_size)
        h, c = state
        gates = (F.linear(x, self.weight_ih, self.bias_ih) + 
                 F.linear(h, self.weight_hh, self.bias_hh))  # (B, 4*hidden_size)
        i, f, g, o = gates.chunk(4, dim=-1)
        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        g = torch.tanh(g)
        o = torch.sigmoid(o)
        c_next = f * c + i * g          # (B, hidden_size)
        h_next = o * torch.tanh(c_next) # (B, hidden_size)
        return h_next, c_next


class MyGRUCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.weight_ih = nn.Parameter(torch.empty(3 * hidden_size, input_size))
        self.weight_hh = nn.Parameter(torch.empty(3 * hidden_size, hidden_size))
        self.bias_ih = nn.Parameter(torch.empty(3 * hidden_size))
        self.bias_hh = nn.Parameter(torch.empty(3 * hidden_size))
        self.reset_parameters()

    def reset_parameters(self):
        stdv = 1.0 / math.sqrt(self.hidden_size)
        for p in self.parameters():
            nn.init.uniform_(p, -stdv, stdv)

    def forward(self, x, h):
        # x: (B, input_size), h: (B, hidden_size)
        x_gates = F.linear(x, self.weight_ih, self.bias_ih)  # (B, 3*h)
        h_gates = F.linear(h, self.weight_hh, self.bias_hh)  # (B, 3*h)
        x_r, x_z, x_n = x_gates.chunk(3, dim=-1)
        h_r, h_z, h_n = h_gates.chunk(3, dim=-1)
        
        r = torch.sigmoid(x_r + h_r)
        z = torch.sigmoid(x_z + h_z)
        n = torch.tanh(x_n + r * h_n)
        h_next = (1.0 - z) * n + z * h  # (B, hidden_size)
        return h_next


In [ ]:
# Assert tests matching PyTorch library cells
torch.manual_seed(42)
B, d, h = 32, 9, 64
x = torch.randn(B, d)
h_init = torch.randn(B, h)
c_init = torch.randn(B, h)

# 1. Test RNN
my_rnn = MyRNNCell(d, h)
lib_rnn = nn.RNNCell(d, h)
lib_rnn.weight_ih.data.copy_(my_rnn.weight_ih.data)
lib_rnn.weight_hh.data.copy_(my_rnn.weight_hh.data)
lib_rnn.bias_ih.data.copy_(my_rnn.bias_ih.data)
lib_rnn.bias_hh.data.copy_(my_rnn.bias_hh.data)
diff_rnn = (my_rnn(x, h_init) - lib_rnn(x, h_init)).abs().max().item()
assert diff_rnn < 1e-5, f"RNN diff too high: {diff_rnn}"

# 2. Test LSTM
my_lstm = MyLSTMCell(d, h)
lib_lstm = nn.LSTMCell(d, h)
lib_lstm.weight_ih.data.copy_(my_lstm.weight_ih.data)
lib_lstm.weight_hh.data.copy_(my_lstm.weight_hh.data)
lib_lstm.bias_ih.data.copy_(my_lstm.bias_ih.data)
lib_lstm.bias_hh.data.copy_(my_lstm.bias_hh.data)
h_my, c_my = my_lstm(x, (h_init, c_init))
h_lib, c_lib = lib_lstm(x, (h_init, c_init))
diff_lstm = max((h_my - h_lib).abs().max().item(), (c_my - c_lib).abs().max().item())
assert diff_lstm < 1e-5, f"LSTM diff too high: {diff_lstm}"

# 3. Test GRU
my_gru = MyGRUCell(d, h)
lib_gru = nn.GRUCell(d, h)
lib_gru.weight_ih.data.copy_(my_gru.weight_ih.data)
lib_gru.weight_hh.data.copy_(my_gru.weight_hh.data)
lib_gru.bias_ih.data.copy_(my_gru.bias_ih.data)
lib_gru.bias_hh.data.copy_(my_gru.bias_hh.data)
diff_gru = (my_gru(x, h_init) - lib_gru(x, h_init)).abs().max().item()
assert diff_gru < 1e-5, f"GRU diff too high: {diff_gru}"

print(f"✓ All custom cells verified! Max differences: RNN={diff_rnn:.1e}, LSTM={diff_lstm:.1e}, GRU={diff_gru:.1e}")


### Parameter Count Derivation ($d = 9, h = 64$)
| Model | Single-Bias Formula | Hand Derived | PyTorch Dual-Bias Formula | PyTorch Count | Discrepancy Reason |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **RNN** | $h(d + h + 1)$ | $64(9 + 64 + 1) = \mathbf{4,736}$ | $h(d + h + 2)$ | $64(9 + 64 + 2) = \mathbf{4,800}$ | PyTorch retains two separate bias vectors ($b_{ih}, b_{hh}$) |
| **LSTM** | $4h(d + h + 1)$ | $4 \times 4,736 = \mathbf{18,944}$ | $4h(d + h + 2)$ | $4 \times 4,800 = \mathbf{19,200}$ | $4 \times 64 = 256$ extra bias parameters across 4 gates |
| **GRU** | $3h(d + h + 1)$ | $3 \times 4,736 = \mathbf{14,208}$ | $3h(d + h + 2)$ | $3 \times 4,800 = \mathbf{14,400}$ | $3 \times 64 = 192$ extra bias parameters across 3 gates |

*Explanation:* PyTorch stores $b_{ih}$ and $b_{hh}$ separately to enable independent GPU kernel execution and cuDNN fusion without an initial vector addition.


## Task 3: UCI HAR Classification Benchmark


In [ ]:
class RecurrentClassifier(nn.Module):
    def __init__(self, cell_type, input_dim=9, hidden_dim=64, num_classes=6):
        super().__init__()
        self.cell_type = cell_type.lower()
        if self.cell_type == 'rnn':
            self.rnn = nn.RNN(input_dim, hidden_dim, batch_first=True)
        elif self.cell_type == 'lstm':
            self.rnn = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        elif self.cell_type == 'gru':
            self.rnn = nn.GRU(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        out, h = self.rnn(x)
        if self.cell_type == 'lstm':
            last_h = h[0][-1]
        else:
            last_h = h[-1]
        return self.fc(last_h)

def train_har_model(model, train_loader, val_loader, epochs=30, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    best_val_acc = -1.0
    best_weights = None
    start_time = time.time()
    for ep in range(epochs):
        model.train()
        for bx, by in train_loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            optimizer.step()
        model.eval()
        val_preds, val_targets = [], []
        with torch.no_grad():
            for bx, by in val_loader:
                preds = model(bx.to(device)).argmax(dim=-1)
                val_preds.extend(preds.cpu().numpy())
                val_targets.extend(by.numpy())
        val_acc = accuracy_score(val_targets, val_preds)
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    sec_per_ep = (time.time() - start_time) / epochs
    if best_weights:
        model.load_state_dict({k: v.to(device) for k, v in best_weights.items()})
    return model, sec_per_ep

def evaluate_har(model, test_loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for bx, by in test_loader:
            p = model(bx.to(device)).argmax(dim=-1)
            preds.extend(p.cpu().numpy())
            targets.extend(by.numpy())
    acc = accuracy_score(targets, preds)
    f1 = f1_score(targets, preds, average='macro')
    return acc, f1, np.array(preds), np.array(targets)


In [ ]:
# Display all Problem 1 Figures from figures/ directory
fig_names = [
    'p1_class_balance.png', 'p1_har_confusion_matrix.png',
    'p1_adding_curves_t200.png', 'p1_adding_mse_vs_t.png',
    'p1_gradient_norms_t100.png', 'p1_hidden_size_sweep.png',
    'p1_ablation_clip_forget.png'
]

for fname in fig_names:
    fpath = os.path.join(FIG_DIR, fname)
    if os.path.exists(fpath):
        plt.figure(figsize=(8, 4.5))
        img = plt.imread(fpath)
        plt.imshow(img)
        plt.axis('off')
        plt.title(fname)
        plt.show()
